In [ ]:
import time
from calendar import monthrange
from datetime import datetime, timedelta
from pathlib import Path

import cdsapi

# Monat auswählen

YEAR = 2015
MONTH = 5

OUTPUT_DIR = (
    Path("/jetfs/scratch/gbrack/MSC/rad_data")
    / f"{YEAR}"
    / f"{MONTH:02d}"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LEVELS = "/".join(str(level) for level in range(1, 138))

# Gewünschte Zielzeiten: erster Tag 00 UTC bis letzter Tag 23 UTC
START = datetime(YEAR, MONTH, 1, 0)
END = datetime(YEAR, MONTH, monthrange(YEAR, MONTH)[1], 23)

c = cdsapi.Client()

# Für 00-06 UTC am Monatsersten: 18 UTC Lauf des Vortags
init_time = START - timedelta(hours=6)

while init_time + timedelta(hours=1) <= END:
    
    # Schritte 1-12 verwenden, beschränkt auf den gewünschten Monat.
    steps = [
        step
        for step in range(1, 13)
        if START <= init_time + timedelta(hours=step) <= END
    ]
    
    first_time = init_time + timedelta(hours=steps[0])
    last_time = init_time + timedelta(hours=steps[-1])
    
    filename = (
        f"TMTTRAD_{first_time:%Y%m%d%H}"
        f"_{last_time:%Y%m%d%H}.nc"
    )
    target = OUTPUT_DIR / filename
    temporary_target = OUTPUT_DIR / f"{filename}.part"
    
    if target.exists():
        print(f"*** Bereits vorhandene: {target.name}", flush=True)
        
    else:
        print(
            f"\nLauf: {init_time:%Y-%m-%d %H} UTC\n"
            f"Zielzeiten: {first_time:%Y-%m-%d %H} bis "
            f"{last_time:%Y-%m-%d %H} UTC\n"
            f"Schritte: {'/'.join(map(str, steps))}",
            flush=True,
        )
        
        start_time = time.monotonic()
        
        c.retrieve(
            "reanalysis-era5-complete",
            {
                "class":     "ea",
                "expver":    "1",
                "stream":    "oper",
                "type":      "fc",
                "levtype":   "ml",
                "levellist": LEVELS,
                "date":      init_time.strftime("%Y-%m-%d"),
                "time":      init_time.strftime("%H:00:00"),
                "step":      "/".join(map(str, steps)),
                "param":     "235001/235002",
                "area":      "90/-179.5/-90/180",
                "grid":      "0.50/0.50",
                "format":    "netcdf"
            },
            str(temporary_target),
        )
        
        # Endgültiger Name wird erst nach erfolgreichem Download vergeben
        temporary_target.replace(target)
        
        elapsed = time.monotonic() - start_time
        print(f"*** {target.name} saved successfully, "
              f"time: {elapsed:.3f}s / {elapsed / 60:.3f} mins ***",
              flush=True,
        )
        
    # Wechsel zwischen 06- und 18-UTC Lauf
    init_time += timedelta(hours=12)
        
print(f"\nMonat {YEAR}-{MONTH:02d} vollständig heruntergeladen.")

2026-10-22 21:05:37,576 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**


*** Bereits vorhandene: TMTTRAD_2015050100_2015050106.nc
*** Bereits vorhandene: TMTTRAD_2015050107_2015050118.nc
*** Bereits vorhandene: TMTTRAD_2015050119_2015050206.nc
*** Bereits vorhandene: TMTTRAD_2015050207_2015050218.nc
*** Bereits vorhandene: TMTTRAD_2015050219_2015050306.nc
*** Bereits vorhandene: TMTTRAD_2015050307_2015050318.nc
*** Bereits vorhandene: TMTTRAD_2015050319_2015050406.nc
*** Bereits vorhandene: TMTTRAD_2015050407_2015050418.nc
*** Bereits vorhandene: TMTTRAD_2015050419_2015050506.nc
*** Bereits vorhandene: TMTTRAD_2015050507_2015050518.nc
*** Bereits vorhandene: TMTTRAD_2015050519_2015050606.nc
*** Bereits vorhandene: TMTTRAD_2015050607_2015050618.nc
*** Bereits vorhandene: TMTTRAD_2015050619_2015050706.nc

Lauf: 2015-05-07 06 UTC
Zielzeiten: 2015-05-07 07 bis 2015-05-07 18 UTC
Schritte: 1/2/3/4/5/6/7/8/9/10/11/12


2026-10-22 21:05:37,765 INFO Request ID is 6b843b83-a7d3-4095-9f47-8df565e32864
2026-10-22 21:05:37,826 INFO status has been updated to accepted


In [ ]:
imopr

In [1]:
import sys

In [2]:
print(sys.executable)

/jetfs/jupyterhub/alpha/envs/magic/bin/python
